# 02 - Limpeza e Enriquecimento dos Dados de Renda por Setor Censitário

Extrai e limpa os dados de renda do IBGE, filtra por Recife e faz join com o shapefile de setores.

In [ ]:
import zipfile
import os
import glob
import pandas as pd
import geopandas as gpd
from IPython.display import display

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()

ROOT        = find_root()
ZIP_PATH    = ROOT / 'data' / 'raw' / 'renda_ibge' / 'Agregados_por_setores_renda_responsavel_BR_20260508_csv.zip'
EXTRACT_DIR = ROOT / 'data' / 'raw' / 'renda_ibge' / 'extraido'
GEO_PATH    = ROOT / 'data' / 'raw' / 'setores_ibge' / 'recife_setores.geojson'
OUTPUT_DIR  = ROOT / 'data' / 'processed'
OUTPUT_PATH = OUTPUT_DIR / 'recife_renda.geojson'

os.makedirs(ZIP_PATH.parent, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'ROOT: {ROOT}')
print(f'Diretório de extração: {EXTRACT_DIR}')
print(f'Diretório de saída:    {OUTPUT_DIR}')

In [ ]:
# Extrai o arquivo ZIP local com os dados de renda do IBGE
csv_existente = glob.glob(os.path.join(EXTRACT_DIR, '**', '*.csv'), recursive=True)

if not csv_existente:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'ZIP de renda não encontrado em: {ZIP_PATH}')

    if not zipfile.is_zipfile(ZIP_PATH):
        raise zipfile.BadZipFile(f'Arquivo ZIP inválido ou corrompido: {ZIP_PATH}')

    print('Extraindo ZIP de renda...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f'Arquivos extraídos em: {EXTRACT_DIR}')
else:
    print(f'Usando CSV de renda já extraído em: {EXTRACT_DIR}')

print('Conteúdo:', os.listdir(EXTRACT_DIR))

In [ ]:
# Localiza o arquivo CSV dentro do diretório extraído (busca recursiva)
csv_files = glob.glob(os.path.join(EXTRACT_DIR, '**', '*.csv'), recursive=True)
if not csv_files:
    raise FileNotFoundError('Nenhum arquivo .csv encontrado após extração.')

CSV_PATH = csv_files[0]
print(f'CSV encontrado: {CSV_PATH}')
if len(csv_files) > 1:
    print(f'  (outros CSVs encontrados: {csv_files[1:]})')

In [ ]:
# Carrega o CSV com pandas; tenta separadores comum do IBGE (;)
print('Carregando CSV de renda...')
try:
    df = pd.read_csv(CSV_PATH, sep=';', dtype=str, encoding='latin-1')
    print(f'Linhas carregadas: {len(df)}')
    print('Colunas:', df.columns.tolist())
except Exception as e:
    print(f'ERRO ao carregar CSV: {e}')
    raise

print('\nPrimeiras linhas:')
display(df.head())

In [ ]:
# Identifica a coluna de código do setor censitário (tipicamente 'Cod_setor' ou 'CD_GEOCODI')
COD_RECIFE = '2611606'

candidatas = [c for c in df.columns if 'setor' in c.lower() or 'geocod' in c.lower() or 'cod' in c.lower()]
print(f'Colunas candidatas para código do setor: {candidatas}')

# Usa a primeira candidata encontrada; ajuste aqui caso necessário
COL_SETOR = candidatas[0] if candidatas else df.columns[0]
print(f'Usando coluna: {COL_SETOR}')

In [ ]:
# Filtra linhas cujos primeiros 7 dígitos do código do setor correspondem a Recife
print(f'Filtrando setores de Recife (primeiros 7 dígitos == {COD_RECIFE})...')
df[COL_SETOR] = df[COL_SETOR].astype(str).str.strip()
df_recife = df[df[COL_SETOR].str[:7] == COD_RECIFE].copy()
print(f'Setores de renda de Recife encontrados: {len(df_recife)}')
display(df_recife.head())

In [ ]:
# Carrega o GeoJSON de setores de Recife gerado no notebook 01
print(f'Carregando GeoJSON de setores: {GEO_PATH}')
try:
    gdf = gpd.read_file(GEO_PATH)
    print(f'Setores no GeoJSON: {len(gdf)}')
    print('Colunas do GeoJSON:', gdf.columns.tolist())
    display(gdf.head(3))
except FileNotFoundError:
    print(f'ERRO: GeoJSON não encontrado em {GEO_PATH}')
    print('Execute o notebook 01_coleta_shapefile.ipynb primeiro.')
    raise
except Exception as e:
    print(f'ERRO ao carregar GeoJSON: {e}')
    raise

In [ ]:
# Identifica a coluna de código do setor no GeoJSON (tipicamente 'CD_SETOR')
geo_candidatas = [c for c in gdf.columns if 'setor' in c.lower() or 'geocod' in c.lower()]
print(f'Colunas candidatas no GeoJSON: {geo_candidatas}')

COL_GEO_SETOR = geo_candidatas[0] if geo_candidatas else 'CD_SETOR'
print(f'Usando coluna do GeoJSON: {COL_GEO_SETOR}')

# Garante que ambos os códigos estão no mesmo formato de string
gdf[COL_GEO_SETOR]   = gdf[COL_GEO_SETOR].astype(str).str.strip()
df_recife[COL_SETOR] = df_recife[COL_SETOR].astype(str).str.strip()

In [ ]:
# Faz o join entre o GeoDataFrame (geometria) e o DataFrame de renda pelo código do setor
print('Realizando join por código do setor censitário...')
gdf_renda = gdf.merge(
    df_recife,
    left_on=COL_GEO_SETOR,
    right_on=COL_SETOR,
    how='left'
)

setores_com_renda = gdf_renda[COL_SETOR].notna().sum()
print(f'Total de setores após join: {len(gdf_renda)}')
print(f'Setores com dados de renda: {setores_com_renda}')
print(f'Setores sem correspondência: {len(gdf_renda) - setores_com_renda}')
display(gdf_renda.head(3))

In [ ]:
# Salva o resultado enriquecido como GeoJSON em data/processed/
print(f'Salvando resultado em: {OUTPUT_PATH}')
try:
    gdf_renda.to_file(OUTPUT_PATH, driver='GeoJSON')
    print('Arquivo salvo com sucesso.')
    print(f'Linhas: {len(gdf_renda)} | Colunas: {len(gdf_renda.columns)}')
except Exception as e:
    print(f'ERRO ao salvar GeoJSON: {e}')
    raise